In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/Real Estate.csv')  
df_clean = df.drop(columns=['country'])
print('Shape:', df_clean.shape)
df_clean.head()

Shape: (4600, 17)


,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip
0,2014-05-02 00:00:00,313000.0,3.0,1.50,1340,7912,1.5,0,0,3,1340,0,1955,2005,18810 Densmore Ave N,Shoreline,WA 98133
1,2014-05-02 00:00:00,2384000.0,5.0,2.50,3650,9050,2.0,0,4,5,3370,280,1921,0,709 W Blaine St,Seattle,WA 98119
2,2014-05-02 00:00:00,342000.0,3.0,2.00,1930,11947,1.0,0,0,4,1930,0,1966,0,26206-26214 143rd Ave SE,Kent,WA 98042
3,2014-05-02 00:00:00,420000.0,3.0,2.25,2000,8030,1.0,0,0,4,1000,1000,1963,0,857 170th Pl NE,Bellevue,WA 98008
4,2014-05-02 00:00:00,550000.0,4.0,2.50,1940,10500,1.0,0,0,4,1140,800,1976,1992,9105 170th Ave NE,Redmond,WA 98052


In [2]:
df_clean.loc[df_clean['price'] == 0, 'price'] = np.nan
df_clean.loc[df_clean['bedrooms'] == 0, 'bedrooms'] = np.nan
df_clean.loc[df_clean['bathrooms'] == 0, 'bathrooms'] = np.nan

print(df_clean.isnull().sum()[['price', 'bedrooms', 'bathrooms']])

price        49
bedrooms      2
bathrooms     2
dtype: int64


In [3]:
df_clean = df_clean.dropna(subset=['price', 'bedrooms', 'bathrooms']).copy()
print(df_clean.shape)

(4549, 17)


In [4]:
print('number of duplicate ', df_clean.duplicated().sum())

number of duplicate  0


In [5]:
df_clean['date'] = pd.to_datetime(df_clean['date'])
print(df_clean['date'].dtype)
print(df_clean['date'].head())

datetime64[ns]
0   2014-05-02
1   2014-05-02
2   2014-05-02
3   2014-05-02
4   2014-05-02
Name: date, dtype: datetime64[ns]


In [6]:
df_clean[['state', 'zipcode']] = df_clean['statezip'].str.split(' ', expand=True)
print(df_clean[['statezip', 'state', 'zipcode']].head())

df_clean = df_clean.drop(columns=['statezip', 'state'])

   statezip state zipcode
0  WA 98133    WA   98133
1  WA 98119    WA   98119
2  WA 98042    WA   98042
3  WA 98008    WA   98008
4  WA 98052    WA   98052


In [7]:
df_clean['house_age'] = df_clean['date'].dt.year - df_clean['yr_built']
print(df_clean[['date', 'yr_built', 'house_age']].head())

        date  yr_built  house_age
0 2014-05-02      1955         59
1 2014-05-02      1921         93
2 2014-05-02      1966         48
3 2014-05-02      1963         51
4 2014-05-02      1976         38


In [10]:
df_clean['was_renovated'] = (df_clean['yr_renovated'] > 0).astype(int)
print(df_clean['was_renovated'].value_counts())

was_renovated
0    2706
1    1843
Name: count, dtype: int64


In [11]:
df_clean['has_basement'] = (df_clean['sqft_basement'] > 0).astype(int)
print(df_clean['has_basement'].value_counts())

has_basement
0    2716
1    1833
Name: count, dtype: int64


In [12]:
print('Final shape:', df_clean.shape)
print('\nMissing values:')
print(df_clean.isnull().sum().sort_values(ascending=False))

print('\nData types:')
print(df_clean.dtypes)

print('\nDuplicate rows:', df_clean.duplicated().sum())

numeric_cols = ['price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'yr_built']
for col in numeric_cols:
    if col in df_clean.columns:
        print(f'\n=== {col} summary ===')
        print(df_clean[col].describe())

invalid = df_clean[
    (df_clean['price'] <= 0) |
    (df_clean['sqft_living'] <= 0) |
    (df_clean['bedrooms'] < 0) |
    (df_clean['bathrooms'] < 0)
]
print(f'\nPotential invalid rows: {len(invalid)}')
if not invalid.empty:
    print(invalid[['price', 'sqft_living', 'bedrooms', 'bathrooms']].head())
else:
    print('No impossible numeric values found.')

Final shape: (4549, 23)

Missing values:
date                       0
price                      0
bedrooms                   0
bathrooms                  0
sqft_living                0
sqft_lot                   0
floors                     0
waterfront                 0
view                       0
condition                  0
sqft_above                 0
sqft_basement              0
yr_built                   0
yr_renovated               0
street                     0
city                       0
zipcode                    0
house_age                  0
price_per_sqft             0
city_avg_price_per_sqft    0
ratio_to_city_avg          0
was_renovated              0
has_basement               0
dtype: int64

Data types:
date                       datetime64[ns]
price                             float64
bedrooms                          float64
bathrooms                         float64
sqft_living                         int64
sqft_lot                            int64
floors        

In [15]:
print("\nPrice range:")
print(df_clean['price'].min(), "to", df_clean['price'].max())

print("\nBedrooms range:")
print(df_clean['bedrooms'].min(), "to", df_clean['bedrooms'].max())

print("\nBathrooms range:")
print(df_clean['bathrooms'].min(), "to", df_clean['bathrooms'].max())

print("\nHouse age range:")
print(df_clean['house_age'].min(), "to", df_clean['house_age'].max())


Price range:
80000.0 to 7062500.0

Bedrooms range:
1.0 to 9.0

Bathrooms range:
0.75 to 8.0

House age range:
0 to 114


In [8]:
df_clean['price_per_sqft'] = df_clean['price'] / df_clean['sqft_living']
city_avg = df_clean.groupby('city')['price_per_sqft'].transform('mean')
df_clean['city_avg_price_per_sqft'] = city_avg
df_clean['ratio_to_city_avg'] = df_clean['price_per_sqft'] / df_clean['city_avg_price_per_sqft']

In [9]:
suspicious = df_clean[(df_clean['ratio_to_city_avg'] > 5) | (df_clean['ratio_to_city_avg'] < 0.2)]
print('Number of outliers ', len(suspicious))
suspicious[['price', 'sqft_living', 'price_per_sqft', 'city', 'city_avg_price_per_sqft', 'ratio_to_city_avg']].sort_values('ratio_to_city_avg', ascending=False)

Number of outliers  8


,price,sqft_living,price_per_sqft,city,city_avg_price_per_sqft,ratio_to_city_avg
4350,26590000.0,1180,22533.898305,Kent,276.574038,81.475103
4346,12899000.0,2190,5889.954338,Seattle,326.397539,18.045339
4348,2199900.0,1120,1964.196429,Covington,192.158042,10.221776
4347,2110000.0,2100,1004.761905,Tukwila,174.401714,5.761193
4349,188000.0,3260,57.668712,Medina,492.127576,0.117182
4398,195000.0,5340,36.516854,Seattle,326.397539,0.111878
4345,84350.0,2630,32.072243,Yarrow Point,490.384531,0.065402
4351,7800.0,780,10.000000,Tukwila,174.401714,0.057339


In [13]:
df_clean = df_clean[
    (df_clean['ratio_to_city_avg'] <= 5) & (df_clean['ratio_to_city_avg'] >= 0.2)
]

In [16]:
df_clean.to_csv(
    'real_estate_cleaned.csv',
    index=False
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
